In [12]:
from agents import OpenAIChatCompletionsModel, AsyncOpenAI
from dotenv import load_dotenv
import os

load_dotenv()


gemini_api_key = os.environ.get('GEMINI_API_KEY')


if not gemini_api_key:
    raise ValueError('Gemini API KEY NOT FOUND')


external_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url='https://generativelanguage.googleapis.com/v1beta/openai'
)


model = OpenAIChatCompletionsModel(
    model = 'gemini-2.5-flash',
    openai_client=external_client
)


In [ ]:
from dataclasses import dataclass
from agents import Agent, Runner, function_tool, RunContextWrapper
import asyncio


@dataclass
class UserInfo:
    name: str
    age: int
    profession: str


@function_tool
def fetch_user_name(wrapper: RunContextWrapper[UserInfo]):
    '''Returns the name of the user'''
    return f"User name is {wrapper.context.name}"

@function_tool
def fetch_user_age(wrapper: RunContextWrapper[UserInfo]):
    '''Returns the age of the user'''
    return f"User age is {wrapper.context.age}"


@function_tool
def fetch_user_bio(wrapper: RunContextWrapper[UserInfo]):
    '''Returns user complete bio details'''
    return f"User name is {wrapper.context.name}, its age is {wrapper.context.age}, and he is {wrapper.context.profession} mainly working on MERN Stack"


# wrapper = RunContextWrapper(context=UserInfo) or wrapper = RunContextWrapper(context=userinfo)
# wrapper = RunContextWrapper(context=UserInfo(name=userinfo.name, age=userinfo.age, profession=userinfo.profession))
agent = Agent(
    name = "Assistant",
    model = model,
    tools=[fetch_user_name,fetch_user_age,fetch_user_bio]
)

userinfo = UserInfo(name="Talha",age=22, profession="Full Stack Developer")


result1 = await Runner.run(starting_agent=agent, input="Complete bio details of user",context=userinfo)
result2 = await Runner.run(starting_agent=agent, input="What is the name of user", context=userinfo)
result3 = await Runner.run(starting_agent=agent, input="What is the age of user", context=userinfo)


print(result1.final_output)
print('-----------------------------------------------------')
print(result2.final_output)
print('-----------------------------------------------------')
print(result3.final_output)

Talha, 22, is a Full Stack Developer mainly working on the MERN Stack.
-----------------------------------------------------
The user name is Talha.
-----------------------------------------------------
The user's age is 22.
